In [1]:
import numpy as np
import random

# Distance matrix (example)
distance_matrix = np.array([
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
])

num_cities = len(distance_matrix)

# Parameters
num_ants = 10
num_iterations = 50
alpha = 1       # pheromone importance
beta = 2        # heuristic importance
evaporation_rate = 0.5
Q = 100         # pheromone deposit factor

# Initialize pheromone
pheromone = np.ones((num_cities, num_cities))

# Visibility (avoid division by zero)
visibility = np.zeros((num_cities, num_cities))
for i in range(num_cities):
    for j in range(num_cities):
        if i != j:
            visibility[i][j] = 1 / distance_matrix[i][j]

# Function to calculate route distance
def route_distance(route):
    distance = 0
    for i in range(len(route) - 1):
        distance += distance_matrix[route[i]][route[i+1]]
    return distance

best_route = None
best_distance = float('inf')

# ACO main loop
for iteration in range(num_iterations):
    all_routes = []
    all_distances = []

    for ant in range(num_ants):
        start = random.randint(0, num_cities - 1)
        route = [start]
        visited = set(route)

        while len(route) < num_cities:
            current = route[-1]
            probabilities = []
            cities = []

            for next_city in range(num_cities):
                if next_city not in visited:
                    tau = pheromone[current][next_city] ** alpha
                    eta = visibility[current][next_city] ** beta
                    prob = tau * eta
                    probabilities.append(prob)
                    cities.append(next_city)

            # Normalize probabilities
            probabilities = np.array(probabilities)
            probabilities = probabilities / probabilities.sum()

            # Roulette wheel selection
            next_city = np.random.choice(cities, p=probabilities)

            route.append(next_city)
            visited.add(next_city)

        # Return to start (complete cycle)
        route.append(route[0])

        distance = route_distance(route)

        all_routes.append(route)
        all_distances.append(distance)

        # Update global best
        if distance < best_distance:
            best_distance = distance
            best_route = route

    # Pheromone evaporation
    pheromone *= (1 - evaporation_rate)

    # Pheromone update
    for i in range(num_ants):
        route = all_routes[i]
        dist = all_distances[i]

        for j in range(len(route) - 1):
            a = route[j]
            b = route[j+1]
            pheromone[a][b] += Q / dist
            pheromone[b][a] += Q / dist  # symmetric

# Output
print("Best Route:", best_route)
print("Shortest Distance:", best_distance)

Best Route: [1, np.int64(0), np.int64(2), np.int64(3), 1]
Shortest Distance: 80
